# Gene-symbol rescue — are the out-of-vocabulary genes really out of vocabulary?

Analysis, not pipeline. Quantifies how many genes scGPT discarded because of a *naming* mismatch
rather than a true absence from its vocabulary.

| | |
|---|---|
| **reads** | `<variant>/SCP542_CCLE.h5ad` — `var_names` per variant |
| | `<variant>/..._oov_genes.csv` — what the embedding step dropped |
| | `reference/hgnc_complete_set.txt`; scGPT's `vocab.json` |
| **writes** | `outputs/embeddings/gene_symbol_rescue.csv` |

**Why it matters for the comparison, not just for tidiness.** Genes dropped here reach the PCA
baseline in full but never reach scGPT. That is an asymmetry between the two arms which has nothing
to do with either representation's quality — it is an artifact of symbol vintage, and it handicaps
one arm only.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.layout import DEFAULT_SCGPT_MODEL_DIR, PipelinePaths
# Derived, not hardcoded (13.08.2026, #25). Resolves to the identical directory; no score is
# involved, since this reads the per-variant OOV tables rather than a targets artifact.
DATA_ROOT = PipelinePaths.build(None, 'hvg5000').processed_dir.parent
# Derived from the path contract (13.08.2026, #25). This was a second copy of the model
# directory, written out in full; layout.DEFAULT_SCGPT_MODEL_DIR already holds it, and
# gen_embeds.py:65 builds the vocab path from it the same way -- so the notebook and the
# script that produced the OOV tables now read the vocabulary from one source instead of two
# that could disagree. Resolves to the identical file.
VOCAB_FILE = DEFAULT_SCGPT_MODEL_DIR / 'vocab.json'
HGNC_FILE = ROOT / 'reference' / 'hgnc_complete_set.txt'
VARIANTS = ['hvg5000', 'all_genes']

# scGPT's vocabulary, minus the three special tokens (<pad>, <cls>, <eoc>).
vocab = {g for g in json.load(open(VOCAB_FILE)) if not g.startswith('<')}

# The OOV tables gen_embeds.py exports alongside each embeddings file: one row per DISCARDED gene,
# already carrying how widely and how strongly it is expressed.
oov = {v: pd.read_csv(DATA_ROOT / v / 'SCP542_CCLE_scGPT_human_embeddings_oov_genes.csv')
       for v in VARIANTS}

hgnc = pd.read_csv(HGNC_FILE, sep='\t', dtype=str, low_memory=False)

print(f'vocabulary        : {len(vocab):,} gene symbols')
print(f'HGNC approved set : {len(hgnc):,} rows, {hgnc.prev_symbol.notna().sum():,} carry a prev_symbol')
for v in VARIANTS:
    print(f'{v:>10} OOV    : {len(oov[v]):,} genes discarded')

vocabulary        : 60,694 gene symbols
HGNC approved set : 45,031 rows, 12,700 carry a prev_symbol
   hvg5000 OOV    : 235 genes discarded
 all_genes OOV    : 1,390 genes discarded


## 1 · The rename map

| | |
|---|---|
| **out** | previous symbol → current symbol, from the HGNC complete set |

A gene absent from scGPT's vocabulary under our symbol may be present under its current one. The map
is built from HGNC's own previous-symbol field rather than from string similarity.

In [2]:
renames = (hgnc[['symbol', 'prev_symbol']]
           .dropna(subset=['prev_symbol'])
           .assign(prev_symbol=lambda d: d['prev_symbol'].str.split('|'))
           .explode('prev_symbol'))
renames['prev_symbol'] = renames['prev_symbol'].str.strip()
renames = renames[renames['prev_symbol'] != '']

# A former symbol HGNC attributes to more than one current gene cannot be resolved by lookup.
counts = renames['prev_symbol'].value_counts()
ambiguous = set(counts[counts > 1].index)

prev_to_current = (renames[~renames['prev_symbol'].isin(ambiguous)]
                   .set_index('prev_symbol')['symbol']
                   .to_dict())

print(f'{len(renames):,} (former -> current) pairs, from {renames["symbol"].nunique():,} current genes')
print(f'{len(ambiguous):,} former symbols map to more than one current gene -> excluded as unresolvable')
print(f'{len(prev_to_current):,} unambiguous rename pairs usable for rescue')

15,879 (former -> current) pairs, from 12,700 current genes
86 former symbols map to more than one current gene -> excluded as unresolvable
15,657 unambiguous rename pairs usable for rescue


> **A second guard is needed and is not here.** Renaming can map two of our rows onto one
> vocabulary entry, or onto a symbol another row already occupies. §3 measures how often that happens;
> a rename that collides is withheld rather than applied.

## 2 · Four-way classification of every discarded gene

| | |
|---|---|
| **out** | each OOV gene labelled: recoverable by rename, genuinely absent, ambiguous, or colliding |

**Why four categories and not a count.** "How many were lost" is not actionable — the recoverable
ones are a fixable defect, the genuinely absent ones are a property of the vocabulary, and the
ambiguous ones are a decision. Collapsing them into one number hides which is which.

In [3]:
CPM_BUDGET = 1_000_000     # CPM is normalized per cell over the full distributed gene set

def classify(gene):
    if gene in prev_to_current:
        return 'rescued' if prev_to_current[gene] in vocab else 'renamed, current symbol also absent'
    if gene in ambiguous:
        return 'unresolvable (former symbol maps to several genes)'
    return 'not an HGNC former symbol'

classified = {}
for v in VARIANTS:
    df = oov[v].copy()
    df['GENE'] = df['GENE'].astype(str)
    n_cells = round(df['n_cells_expressed'].iloc[0] / df['pct_cells_expressed'].iloc[0] * 100)

    df['outcome'] = df['GENE'].map(classify)
    df['current_symbol'] = df['GENE'].map(prev_to_current)
    classified[v] = df

    summary = (df.groupby('outcome')
                 .agg(genes=('GENE', 'size'),
                      cpm_per_cell=('total_expression', lambda s: s.sum() / n_cells),
                      expressed_in_over_10pct_cells=('pct_cells_expressed', lambda s: (s > 10).sum()))
                 .sort_values('cpm_per_cell', ascending=False))
    summary['pct_of_transcriptome'] = 100 * summary['cpm_per_cell'] / CPM_BUDGET

    print(f'=== {v}: {len(df):,} discarded genes, n_cells = {n_cells:,}')
    print(summary.round(3).to_string())
    print()

=== hvg5000: 235 discarded genes, n_cells = 53,513
                                     genes  cpm_per_cell  expressed_in_over_10pct_cells  pct_of_transcriptome
outcome                                                                                                      
not an HGNC former symbol              231       590.019                             12                 0.059
renamed, current symbol also absent      4         5.692                              0                 0.001

=== all_genes: 1,390 discarded genes, n_cells = 53,513
                                                    genes  cpm_per_cell  expressed_in_over_10pct_cells  pct_of_transcriptome
outcome                                                                                                                     
not an HGNC former symbol                            1348      2104.799                             50                 0.210
unresolvable (former symbol maps to several genes)      3       428.981       

## 3 · Collisions

| | |
|---|---|
| **out** | rescued symbols whose current name already exists as its own row |

Applying such a rename would merge two distinct measurements. The rename is withheld, and the count
is reported so the cost of withholding is visible rather than assumed to be zero.

In [4]:
import scanpy as sc

for v in VARIANTS:
    existing = set(sc.read_h5ad(DATA_ROOT / v / 'SCP542_CCLE.h5ad', backed='r').var_names)
    rescued = classified[v][classified[v]['outcome'] == 'rescued']
    collide = rescued[rescued['current_symbol'].isin(existing)]

    print(f'=== {v}: {len(rescued):,} rescued, {len(collide):,} collide with an existing row')
    if len(collide):
        print(collide[['GENE', 'current_symbol', 'pct_cells_expressed', 'total_expression']]
              .sort_values('total_expression', ascending=False).to_string(index=False))
    print()

=== hvg5000: 0 rescued, 0 collide with an existing row

=== all_genes: 13 rescued, 11 collide with an existing row
      GENE current_symbol  pct_cells_expressed  total_expression
HNRNPU-AS1         HNRNPU            12.380169     441833.490021
  C10orf12           LCOR             8.693215     268306.504719
    CTAGE5           MIA2             5.166969     153368.922297
   C2orf48           RRM2             3.328163      99804.012297
   TMEM133       ARHGAP42             2.141536      59137.152671
   MICALCL         MICAL2             1.984564      53641.034651
  KIAA1107          BTBD8             1.207183      35196.393168
UBXN10-AS1        PLA2G2C             0.512025      14364.149180
   C9orf47          S1PR3             0.338236      10211.438118
 C10orf113           NEBL             0.084092       2931.700073
 LINC00444         SUCLA2             0.082223       2834.296639



## 4 · The artifact

| | |
|---|---|
| **out** | `outputs/embeddings/gene_symbol_rescue.csv` — one row per discarded gene with its class |

Read alongside the OOV summary the embedding step writes: this table says *why* each gene is missing,
which that summary cannot.

In [5]:
OUT = ROOT / 'notebooks' / 'outputs' / 'embeddings' / 'gene_symbol_rescue.csv'

frames = []
for v in VARIANTS:
    existing = set(sc.read_h5ad(DATA_ROOT / v / 'SCP542_CCLE.h5ad', backed='r').var_names)
    df = classified[v][['GENE', 'outcome', 'current_symbol', 'n_cells_expressed',
                        'pct_cells_expressed', 'total_expression']].copy()
    df.insert(0, 'variant', v)
    # only meaningful where a current symbol exists; False elsewhere rather than NaN.
    df['collides_with_existing_row'] = df['current_symbol'].isin(existing) & df['current_symbol'].notna()
    frames.append(df)

artifact = (pd.concat(frames, ignore_index=True)
              .sort_values(['variant', 'total_expression'], ascending=[True, False]))
OUT.parent.mkdir(parents=True, exist_ok=True)
artifact.to_csv(OUT, index=False)

print(f'wrote {OUT.relative_to(ROOT)}  ({len(artifact):,} rows)')
print(artifact.groupby(['variant', 'outcome']).size().to_string())

wrote notebooks/outputs/embeddings/gene_symbol_rescue.csv  (1,625 rows)
variant    outcome                                           
all_genes  not an HGNC former symbol                             1348
           renamed, current symbol also absent                     26
           rescued                                                 13
           unresolvable (former symbol maps to several genes)       3
hvg5000    not an HGNC former symbol                              231
           renamed, current symbol also absent                      4
